# VCB 13U House Draft Analysis
This notebook is Google Colab-compatible. Upload `VCB House - 13u PeeWee Assessment.xlsx`, then run the cells in order.

In [ ]:
!pip -q install pandas numpy matplotlib scikit-learn openpyxl nbformat

## 1. Introduction
This workflow inspects the workbook, cleans the data, builds `master_players`, creates rankings and tiers, and simulates a snake draft.

In [ ]:
from pathlib import Path
import pandas as pd
from analysis_pipeline import (
    NUM_TEAMS, ROSTER_SIZE, RANDOM_SEED, DRAFT_STRATEGY,
    WEIGHT_OVERALL, WEIGHT_PITCHING, WEIGHT_BALANCE, WEIGHT_UPSIDE,
    inspect_workbook, build_master_players, build_rankings,
    build_hidden_value_table, build_draft_board, simulate_draft,
    save_visualizations, build_report
)
WORKBOOK_PATH = Path('VCB House - 13u PeeWee Assessment.xlsx')

## 2. Load workbook

In [ ]:
inspection, inspection_summary = inspect_workbook(WORKBOOK_PATH)
inspection_summary

## 3. Inspect workbook

In [ ]:
inspection['cross_sheet_summary']

## 4. Clean data

In [ ]:
master_players = build_master_players(WORKBOOK_PATH)
master_players.head()

## 5. Build master player table

In [ ]:
master_players.shape, master_players[['player_name','overall_rank','pitcher_rank','tier']].head(10)

## 6. Generate rankings

In [ ]:
rankings = build_rankings(master_players)
rankings['overall'][['player_name','overall_rank','overall_composite']].head(20)

## 7. Create tiers

In [ ]:
master_players['tier'].value_counts().sort_index()

## 8. Identify hidden-value players

In [ ]:
hidden_value_players = build_hidden_value_table(master_players)
hidden_value_players.head(20)

## 9. Build draft board

In [ ]:
draft_board = build_draft_board(master_players)
draft_board.head(20)

## 10. Configure simulator

In [ ]:
# Editable parameters
NUM_TEAMS = 6
ROSTER_SIZE = 12
RANDOM_SEED = 13
DRAFT_STRATEGY = 'both'  # 'best_available', 'balanced', or 'both'
WEIGHT_OVERALL = 0.55
WEIGHT_PITCHING = 0.20
WEIGHT_BALANCE = 0.15
WEIGHT_UPSIDE = 0.10
LOCKED_PLAYERS = {}

## 11. Simulate snake draft

In [ ]:
best_available_results, best_available_team_strength = simulate_draft(
    draft_board, master_players, num_teams=NUM_TEAMS, roster_size=ROSTER_SIZE, strategy='best_available', locked_players=LOCKED_PLAYERS,
    weight_overall=WEIGHT_OVERALL, weight_pitching=WEIGHT_PITCHING, weight_balance=WEIGHT_BALANCE, weight_upside=WEIGHT_UPSIDE
)
balanced_results, balanced_team_strength = simulate_draft(
    draft_board, master_players, num_teams=NUM_TEAMS, roster_size=ROSTER_SIZE, strategy='balanced', locked_players=LOCKED_PLAYERS,
    weight_overall=WEIGHT_OVERALL, weight_pitching=WEIGHT_PITCHING, weight_balance=WEIGHT_BALANCE, weight_upside=WEIGHT_UPSIDE
)
best_available_results.head(12)

## 12. Evaluate team balance

In [ ]:
best_available_team_strength, balanced_team_strength

## 13. Visualize team strength

In [ ]:
save_visualizations(master_players, {'best_available': best_available_team_strength, 'balanced': balanced_team_strength}, Path('.'))

## 14. Export outputs

In [ ]:
rankings['overall'].to_csv('player_rankings_overall.csv', index=False)
rankings['pitchers'].to_csv('player_rankings_pitchers.csv', index=False)
master_players.to_csv('cleaned_master_players.csv', index=False)
master_players[['player_name','tier','overall_rank','overall_composite']].to_csv('player_tiers.csv', index=False)
hidden_value_players.to_csv('hidden_value_players.csv', index=False)
draft_board.to_csv('draft_board.csv', index=False)
pd.concat([best_available_results, balanced_results], ignore_index=True).to_csv('simulated_draft_results.csv', index=False)
pd.concat([
    best_available_team_strength.assign(strategy='best_available'),
    balanced_team_strength.assign(strategy='balanced')
], ignore_index=True).to_csv('team_strength_summary.csv', index=False)

## 15. Final summary

In [ ]:
report = build_report(inspection_summary, master_players, hidden_value_players, {'best_available': best_available_team_strength, 'balanced': balanced_team_strength})
print(report)